In [ ]:
import json
import requests
import openai

client = openai.OpenAI()
messages = []

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


In [ ]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다. 인기 영화를 추천하거나 보여줄 때 사용하세요.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 영화 ID에 해당하는 영화의 상세 정보(제목, 줄거리, 평점 등)를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "정보를 가져올 영화의 고유 ID",
                    },
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "특정 영화 ID에 해당하는 영화의 출연진(배우) 및 제작진 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "출연진 정보를 가져올 영화의 고유 ID",
                    },
                },
                "required": ["id"],
            },
        },
    },
]

In [ ]:
def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    choice = response.choices[0]

    # 모델이 도구 호출을 요청한 경우
    if choice.finish_reason == "tool_calls":
        tool_calls = choice.message.tool_calls
        messages.append(choice.message)  # assistant 메시지(tool_calls 포함) 추가

        for tool_call in tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            print(f"[도구 호출] {fn_name}({fn_args})")

            fn_result = FUNCTION_MAP[fn_name](**fn_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(fn_result, ensure_ascii=False),
            })

        # 도구 결과를 바탕으로 최종 응답 생성
        call_ai()
    else:
        message = choice.message.content
        messages.append({"role": "assistant", "content": message})
        print(f"AI: {message}")


In [ ]:
while True:
    user_input = input("메시지를 입력하세요 (종료: q): ")
    if user_input.lower() == "q":
        break
    print(f"User: {user_input}")
    messages.append({"role": "user", "content": user_input})
    call_ai()

In [ ]:
'''User: 
AI: 안녕하세요! 무엇을 도와드릴까요?
User: "지금 인기 있는 영화가 무엇인지 알려줘"
[도구 호출] get_popular_movies({})
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Obsession**
   - ![Obsession](https://image.tmdb.org/t/p/w780/2G249T4Sgu8gXIZpaXWnxHYYNQV.jpg)
   - **개요**: 신비로운 "원 위시 윌로우"를 깨트린 후, 한 사랑에 빠진 청년이 원하는 것을 얻게 되지만 그 desire가 어두운 대가를 치르게 됨을 알게 됩니다.
   - **평점**: 7.9
   - **개봉일**: 2026-05-13

2. **Peddi**
   - ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)
   - **개요**: 1980년대 안드라 프라데시의 지방 마을에서, 한 spirited villager가 자신의 자존심을 지키기 위해 공동체를 스포츠로 단결시킵니다.
   - **평점**: 6.3
   - **개봉일**: 2026-06-03

3. **Lee Cronin's The Mummy**
   - ![Lee Cronin's The Mummy](https://image.tmdb.org/t/p/w780/1q308iixueCU4pFtSYugNOevtNx.jpg)
   - **개요**: 기자의 어린 딸이 사라진 후, 8년 만에 돌아오지만, 기쁜 재회가 살아있는 악몽으로 변해버립니다.
   - **평점**: 8.1
   - **개봉일**: 2026-04-15

4. **The Mandalorian and Grogu**
   - ![The Mandalorian and Grogu](https://image.tmdb.org/t/p/w780/5Vi8dSauVwH1HOsiZceDMbRr1Ca.jpg)
   - **개요**: 악당 제국이 무너진 후, 신생 공화국은 전설적인 Mandalorian 보상 사냥꾼 Din Djarin의 도움을 요청합니다.
   - **평점**: 6.8
   - **개봉일**: 2026-05-20

5. **Kara**
   - ![Kara](https://image.tmdb.org/t/p/w780/6U6i4qhgHR1MWkUb6OGQwNpqcZC.jpg)
   - **개요**: 한 도둑이 정직하게 살려고 하지만, 아버지를 담보로 걸고 있는 은행들의 함정에 빠져 범죄에 다시 발을 디딥니다.
   - **평점**: 6.3
   - **개봉일**: 2026-04-30

더 궁금한 영화가 있으신가요?
User: 
AI: 더 필요한 정보가 있으면 말씀해 주세요! 어떤 도움이 필요하신가요?
User: movie ID 550에 해당하는 영화가 무엇인지 알려줘
[도구 호출] get_movie_details({'id': 550})
AI: 영화 ID 550에 해당하는 영화의 정보는 다음과 같습니다:

### **Fight Club**
- ![Fight Club](https://image.tmdb.org/t/p/w780/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg)
- **개요**: 불면증에 시달리는 남자와 미끄러운 비누 판매자가 본능적인 남성 공격성을 새로운 형태의 치료로 전환합니다. 그들의 개념은 금지된 "파이트 클럽"이 각 도시에서 형성되는 결과를 낳지만, 한 괴상한 인물이 그 길을 방해하며 통제할 수 없는 혼란으로 나아갑니다.
- **장르**: 드라마, 스릴러
- **개봉일**: 1999-10-15
- **러닝타임**: 139분
- **예산**: $63,000,000
- **수익**: $100,853,753
- **평점**: 8.4 (투표 수: 32,112)
- **태그라인**: Mischief. Mayhem. Soap.
- **공식 웹사이트**: [여기](https://www.20thcenturystudios.com/movies/fight-club)

더 궁금한 부분이 있으면 말씀해 주세요!
User: movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘
[도구 호출] get_movie_credits({'id': 550})
AI: 영화 **Fight Club**에 출연한 주요 배우들은 다음과 같습니다:

1. **Edward Norton** - Narrator  
   ![Edward Norton](https://image.tmdb.org/t/p/w185/8nytsqL59SFJTVYVrN72k6qkGgJ.jpg)

2. **Brad Pitt** - Tyler Durden  
   ![Brad Pitt](https://image.tmdb.org/t/p/w185/m09Y1YfPPeNYYUSHnnVqahkrC1o.jpg)

3. **Helena Bonham Carter** - Marla Singer  
   ![Helena Bonham Carter](https://image.tmdb.org/t/p/w185/hJMbNSPJ2PCahsP3rNEU39C8GWU.jpg)

4. **Meat Loaf** - Robert Paulson  
   ![Meat Loaf](https://image.tmdb.org/t/p/w185/1zkohpaG3my4qQAZGVgzgPuXwZ6.jpg)

5. **Jared Leto** - Angel Face  
   ![Jared Leto](https://image.tmdb.org/t/p/w185/ca3x0OfIKbJppZh8S1Alx3GfUZO.jpg)

6. **Zach Grenier** - Richard Chesler (Regional Manager)  
   ![Zach Grenier](https://image.tmdb.org/t/p/w185/fSyQKZO39sUsqY283GXiScOg3Hi.jpg)

7. **Holt McCallany** - The Mechanic  
   ![Holt McCallany](https://image.tmdb.org/t/p/w185/iRo9YUNMwZg4UCq7dapo0HydDmI.jpg)

8. **Eion Bailey** - Ricky  
   ![Eion Bailey](https://image.tmdb.org/t/p/w185/3DW13W47cKk4LQZwS4EvRaNBoVu.jpg)

이 외에도 많은 배우들이 출연했습니다. 더 궁금한 정보가 있으면 말씀해 주세요!
'''